# scvi/02 — Project Query (Pure Inference, No Retraining)

Loads the saved SCANVI reference model and projects new query cells by
passing them through the **frozen encoder** — no gradient updates, no
retraining epochs. This mirrors what Seurat CCA `MapQuery` does.

**API used:**
```python
scanvi_model = scvi.model.SCANVI.load(model_dir)          # load once
latent  = scanvi_model.get_latent_representation(adata_q)  # encoder forward pass
labels  = scanvi_model.predict(adata_q)                    # classifier forward pass
```

> **scArches (surgery) alternative:** for out-of-distribution data, brief fine-tuning
> via `SCANVI.load_query_data(adata_q, model)` + `model.train(max_epochs=100)` is more
> robust. Switch the `USE_SURGERY` flag below to compare.

**Inputs:** `data/scvi/query.h5ad`, `models/scanvi_reference/`  
**Outputs:** `data/scvi/query_projected.h5ad`

In [ ]:
# Parameters — injected by papermill
query_h5ad            = "data/scvi/query.h5ad"
reference_h5ad_trained= "data/scvi/reference_trained.h5ad"
model_dir             = "models/scanvi_reference"
output_projected_h5ad = "data/scvi/query_projected.h5ad"
labels_key            = "cluster"
USE_SURGERY           = False  # True = scArches brief fine-tune; False = pure inference
n_epochs_surgery      = 100    # used only if USE_SURGERY=True


## 0 · Import

In [ ]:
from IPython.display import display as ipy_display, Image
import io
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

import anndata as ad
import scanpy as sc
import scvi
import os, time
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

scvi.settings.seed = 42
t_nb_start = time.time()
print(f"scvi-tools {scvi.__version__}")


## 1 · Load Saved Model

In [ ]:
print(f"Loading SCANVI model from {model_dir} ...")
t0 = time.time()
# save_anndata=True was used at save time, so reference AnnData is embedded
scanvi_model = scvi.model.SCANVI.load(model_dir)
t_load = time.time() - t0
print(f"  loaded in {t_load:.1f}s")
print(f"  gene set : {len(scanvi_model.adata.var_names):,} genes")
print(f"  labels   : {sorted(scanvi_model.adata.obs[labels_key].unique())}")


## 2 · Load Query & Align Gene Set

Query must contain the same genes the model was trained on.
Extra query genes are dropped; genes missing from the query are zero-filled.

In [ ]:
adata_query = ad.read_h5ad(query_h5ad)
print(f"Query raw: {adata_query.n_obs:,} cells x {adata_query.n_vars:,} genes")

# Align to reference gene set
ref_genes  = scanvi_model.adata.var_names
shared     = adata_query.var_names.intersection(ref_genes)
print(f"Shared genes with reference: {len(shared):,} / {len(ref_genes):,}")

if len(shared) < len(ref_genes):
    import scipy.sparse, numpy as np
    # Zero-fill missing genes
    import anndata as ad2
    missing = ref_genes.difference(adata_query.var_names)
    zero_block = scipy.sparse.csr_matrix((adata_query.n_obs, len(missing)))
    extra = ad.AnnData(
        X   = zero_block,
        obs = adata_query.obs,
        var = pd.DataFrame(index=missing)
    )
    adata_query = ad.concat([adata_query, extra], axis=1)

# Reorder to match reference exactly
adata_query = adata_query[:, ref_genes].copy()
# Ensure counts layer exists
if "counts" not in adata_query.layers:
    adata_query.layers["counts"] = adata_query.X.copy()
print(f"Query aligned: {adata_query.n_obs:,} cells x {adata_query.n_vars:,} genes")


## 3 · Project Query

**Pure inference:** encoder and classifier weights are frozen.
**Surgery (optional):** brief fine-tune to adapt encoder to query distribution.

In [ ]:
if USE_SURGERY:
    print("=== scArches surgery mode (brief fine-tune) ===")
    t_proj_start = time.time()
    surgery_model = scvi.model.SCANVI.load_query_data(
        adata_query, scanvi_model,
        freeze_expression=True,
    )
    surgery_model.train(
        max_epochs=n_epochs_surgery,
        plan_kwargs={"weight_decay": 0.0},
    )
    t_proj_elapsed = time.time() - t_proj_start
    active_model = surgery_model
    print(f"Surgery fine-tune: {t_proj_elapsed:.1f}s")
else:
    print("=== Pure inference mode (frozen model) ===")
    t_proj_start = time.time()
    active_model = scanvi_model  # use reference model directly
    t_proj_elapsed = 0.0

# Latent representation
t0 = time.time()
adata_query.obsm["X_scANVI"] = active_model.get_latent_representation(adata_query)
t_latent = time.time() - t0

# Label prediction
t0 = time.time()
adata_query.obs["predicted_label"] = active_model.predict(adata_query)
t_predict = time.time() - t0

print(f"get_latent_representation : {t_latent:.2f}s")
print(f"predict                   : {t_predict:.2f}s")
print(f"\nPredicted label distribution:")
print(adata_query.obs["predicted_label"].value_counts().to_string())


## 4 · UMAP in scANVI Latent Space

In [ ]:
# ── Project query onto reference UMAP via sc.tl.ingest ─────────────────────
print(f"Loading trained reference: {reference_h5ad_trained}")
adata_ref = ad.read_h5ad(reference_h5ad_trained)

sc.tl.ingest(adata_query, adata_ref, obs=labels_key, embedding_method="umap")
print(f"Query projected onto reference UMAP: {adata_query.obsm['X_umap'].shape}")

# Joint UMAP: reference (left) + query projected (right)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sc.pl.umap(adata_ref, color=labels_key,
           title="Reference", ax=axes[0], show=False,
           legend_loc="right margin", legend_fontsize=8)
sc.pl.umap(adata_query, color="predicted_label",
           title="Query (projected onto reference UMAP)",
           ax=axes[1], show=False,
           legend_loc="right margin", legend_fontsize=8)
plt.tight_layout()
buf = io.BytesIO()
fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
buf.seek(0)
ipy_display(Image(data=buf.read()))
plt.close(fig)


## 5 · Identify CD8+ T Cells

In [ ]:
# CD8 detection
cd8_mask = adata_query.obs["predicted_label"].str.contains(
    "CD8|cd8|exhausted|effector|TRM", case=False, na=False
)
adata_query.obs["is_cd8"] = cd8_mask
n_cd8 = int(cd8_mask.sum())
print(f"CD8+ T cells: {n_cd8:,} / {adata_query.n_obs:,} ({n_cd8/adata_query.n_obs*100:.1f}%)")

# Highlight CD8+ cells in reference UMAP space (mirrors CCA DimPlot cells.highlight)
import numpy as np
umap_coords = adata_query.obsm["X_umap"]
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(umap_coords[~cd8_mask, 0], umap_coords[~cd8_mask, 1],
           c="lightgrey", s=2, rasterized=True, label="non-CD8")
ax.scatter(umap_coords[cd8_mask, 0], umap_coords[cd8_mask, 1],
           c="#e63946", s=4, rasterized=True, label=f"CD8+ (n={n_cd8})")
ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
ax.set_title(f"CD8+ T cells highlighted (n={n_cd8})")
ax.legend(markerscale=3, frameon=False, fontsize=9)
ax.axis("off")
plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=120, bbox_inches='tight'); buf.seek(0)
ipy_display(Image(data=buf.read())); plt.close(fig)


## 6 · Evaluate Label Transfer


In [ ]:
# ── Accuracy evaluation (requires ground-truth labels in query) ─────────────
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix

gt_col = labels_key  # 'cluster' — original cell-type labels in query adata

if gt_col in adata_query.obs.columns:
    y_true  = adata_query.obs[gt_col].astype(str)
    y_pred  = adata_query.obs['predicted_label'].astype(str)
    classes = sorted(set(y_true) | set(y_pred))

    acc = (y_true == y_pred).mean()
    print(f'Overall label transfer accuracy: {acc:.1%}')

    # Confusion matrix heatmap
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                ax=ax, linewidths=0.5, linecolor='grey')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('scVI/scANVI label transfer — confusion matrix')
    plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    buf.seek(0)
    ipy_display(Image(data=buf.read()))
    plt.close(fig)

    # ── CD8_ex precision / recall (CD8_ex = positive class) ──────────────
    pos_class  = 'CD8_ex'
    y_true_bin = (y_true == pos_class)
    y_pred_bin = (y_pred == pos_class)
    tp = int((y_true_bin  &  y_pred_bin).sum())
    fp = int((~y_true_bin &  y_pred_bin).sum())
    fn = int((y_true_bin  & ~y_pred_bin).sum())
    prec = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    rec  = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else float('nan')

    print(f'\n=== CD8_ex (positive class) precision / recall ===')
    print(f'  True positives  (TP): {tp}')
    print(f'  False positives (FP): {fp}')
    print(f'  False negatives (FN): {fn}')
    print(f'  Precision : {prec:.3f}')
    print(f'  Recall    : {rec:.3f}')
    print(f'  F1        : {f1:.3f}')
else:
    print(f"Ground-truth column '{gt_col}' not found in query — skipping evaluation.")


## 7 · Save Projected Query


In [ ]:
os.makedirs(os.path.dirname(output_projected_h5ad), exist_ok=True)
adata_query.write_h5ad(output_projected_h5ad)
print(f"Saved -> {output_projected_h5ad}")
print(f"Key obs columns: {list(adata_query.obs.columns)}")


## 8 · Timing Summary


In [ ]:
t_total = time.time() - t_nb_start
mode_str = 'scArches surgery' if USE_SURGERY else 'pure inference'
print("=== scvi/02 Timing Summary ===")
print(f"  Mode                         : {mode_str}")
if USE_SURGERY:
    print(f"  Surgery fine-tune ({n_epochs_surgery} epochs): {t_proj_elapsed:7.1f}s  ({t_proj_elapsed/60:5.1f} min)")
print(f"  get_latent_representation    : {t_latent:7.3f}s")
print(f"  predict (label transfer)     : {t_predict:7.3f}s")
print(f"  Total NB elapsed             : {t_total:7.1f}s  ({t_total/60:5.1f} min)")
print()
print("Compare vs Seurat CCA (from NB02 Timing Summary):")
print("  CCA FindTransferAnchors is typically >> scANVI pure inference")
print("  (CCA scales O(n²) with cells; scANVI inference is O(n) / linear)")
